# ddqn-router — Colab quickstart

Runs end-to-end on a free Colab CPU in **under 5 minutes**.

1. Install `ddqn-router` from PyPI.
2. Scaffold a project with `ddqn-router init`.
3. Generate 60 synthetic queries with a deterministic rule-based function (no API key needed).
4. Optionally label them with `gpt-4o-mini` from Colab secrets; otherwise fall back to a mock labeler.
5. Split, train for 20k steps on CPU.
6. Route 3 example queries and print `router.explain()`.

## 1. Install

In [ ]:
%pip install -q ddqn-router

## 2. Scaffold

In [ ]:
!ddqn-router init --path /content/demo --force
%cd /content/demo

## 3. Generate 60 synthetic queries (rule-based, deterministic)

In [ ]:
import random

TEMPLATES = {
    0: [
        "invoice double charge",
        "refund my payment",
        "cancel my subscription",
        "billing statement wrong",
    ],
    1: ["api 500 error", "webhook broken", "integration bug", "sdk crash on import"],
    2: [
        "reset my password",
        "change account email",
        "locked out of account",
        "update my permissions",
    ],
}
COMBOS = [[0], [1], [2], [0, 2], [1, 2], [0, 1]]

rng = random.Random(42)
queries = []
for _ in range(60):
    agents = rng.choice(COMBOS)
    text = " and ".join(rng.choice(TEMPLATES[a]) for a in agents)
    queries.append(text)

with open("data/queries.txt", "w") as f:
    for q in queries:
        f.write(q + "\n")
print(f"Wrote {len(queries)} queries to data/queries.txt")

## 4. Label the queries

If you have an OpenAI API key in Colab secrets (name `OPENAI_API_KEY`), the LLM labeler will run.
Otherwise a deterministic mock labeler is used so the notebook still works end-to-end without a key.

In [ ]:
import json
import os

api_key = None
try:
    from google.colab import userdata

    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")

if api_key:
    os.environ["DDQN_ROUTER_API_KEY"] = api_key
    !ddqn-router label --config config.yaml --input data/queries.txt --output data/tasks.jsonl
else:
    print("No API key found — using mock labeler")
    KEYWORDS = {
        0: ["invoice", "refund", "billing", "subscription", "payment"],
        1: ["api", "webhook", "integration", "sdk", "crash", "bug", "error"],
        2: ["password", "account", "email", "locked", "permissions"],
    }
    os.makedirs("data", exist_ok=True)
    with open("data/tasks.jsonl", "w") as out:
        for i, q in enumerate(queries):
            agents = [aid for aid, kws in KEYWORDS.items() if any(k in q.lower() for k in kws)]
            if not agents:
                agents = [0]
            out.write(
                json.dumps({"id": f"ex_{i:03d}", "text": q, "required_agents": sorted(agents)})
                + "\n"
            )
    print("Wrote data/tasks.jsonl")

## 5. Split & train (20k steps)

In [ ]:
!ddqn-router dataset split --input data/tasks.jsonl --output-dir data/

import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["training"]["total_steps"] = 20000
with open("config.yaml", "w") as f:
    yaml.safe_dump(cfg, f)

!ddqn-router train --config config.yaml

## 6. Route 3 example queries

In [ ]:
from ddqn_router import DDQNRouter

router = DDQNRouter.load("./artifacts/")

for q in [
    "my invoice was charged twice",
    "api 500 error and password reset",
    "locked out, billing statement wrong too",
]:
    result = router.route(q)
    print(
        f"{q!r}\n -> {result.agent_names}  (conf={result.confidence:.2f}, steps={result.steps})\n"
    )

router.explain("api 500 error and password reset")